# Segmentación de Jugadores de Campo FIFA19

## 1. Objetivo del Proyecto

El objetivo es **identificar segmentos (clusters) de jugadores de campo** (excluyendo porteros) que compartan características técnicas y físicas similares, basándonos únicamente en habilidades puras, sin usar métricas compuestas como Overall, Potential o Special. Esto permitirá construir equipos ficticios balanceados combinando diferentes perfiles tácticos.

**¿Por qué excluir métricas compuestas?** Porque estas ya resumen el rendimiento general y sesgarían la segmentación hacia una única dimensión de "calidad", ocultando perfiles especializados (por ejemplo, un defensor con bajo Overall pero excelente en entrada y fuerza).

## 2. Importación de librerías y configuración inicial

Cargamos las librerías necesarias y fijamos una semilla (`333`) para garantizar la reproducibilidad de todos los procesos aleatorios (PCA, KMeans, etc.).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import altair as alt

# Semillas para reproducibilidad
RANDOM_SEED = 333
np.random.seed(RANDOM_SEED)

# Configuración de gráficos
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
alt.theme.enable('dark')

ThemeRegistry.enable('dark')

## 3. Carga de datos y selección de variables

Cargamos el dataset `FIFA19-DS.csv` (más de 17,000 registros y 89 columnas). Las variables se seleccionaron basándose en un análisis exploratorio previo (EDA) y en el diccionario de variables proporcionado.

**Criterios de selección:**
- **Variables demográficas**: Age (la edad influye en atributos físicos y desarrollo).
- **Habilidades especiales**: Weak Foot, Skill Moves, Work Rate (versatilidad y compromiso táctico).
- **Físicas**: Height_cm, Weight_kg (dimensiones corporales, convertidas a unidades métricas).
- **Técnicas**: 17 variables que cubren aspectos ofensivos (ej. Finishing, Dribbling) y defensivos (ej. Marking, StandingTackle).
- **Físico-mentales**: Acceleration, SprintSpeed, Stamina, Composure, etc. (velocidad, resistencia, inteligencia de juego).

**Variables excluidas explícitamente:** Overall, Potential, Special (compuestas), Value, Wage, International Reputation (mercado), Position (categórica, usada solo en perfilamiento), Preferred Foot, Body Type (categóricas no numéricas), y todas las variables específicas de porteros (GK*).

In [2]:
# Cargar datos
df = pd.read_csv('Datos/FIFA19-DS.csv', encoding='utf-8')
print(f"Dimensiones originales del dataset: {df.shape}")

# Lista de variables seleccionadas para clustering (jugadores de campo)
selected_vars = [
    'Age', 'Weak Foot', 'Skill Moves', 'Work Rate',
    'Height_cm', 'Weight_kg',
    'Crossing', 'Finishing', 'HeadingAccuracy', 'ShortPassing', 'Volleys',
    'Dribbling', 'Curve', 'FKAccuracy', 'LongPassing', 'BallControl',
    'ShotPower', 'LongShots', 'Penalties', 'Marking', 'StandingTackle',
    'SlidingTackle', 'Interceptions',
    'Acceleration', 'SprintSpeed', 'Agility', 'Reactions', 'Balance',
    'Jumping', 'Stamina', 'Strength', 'Aggression', 'Positioning',
    'Vision', 'Composure'
]
print(f"Total de variables seleccionadas: {len(selected_vars)}")

Dimensiones originales del dataset: (17140, 76)
Total de variables seleccionadas: 35


## 4. Preprocesamiento de datos

### 4.1. Exclusión de porteros

Los porteros tienen atributos especiales (GK Diving, GK Handling, etc.) que no son comparables con los jugadores de campo. Además, sus valores en variables como Finishing o Dribbling son muy bajos y distorsionarían los clusters. Por tanto, filtramos eliminando todas las filas con `Position == 'GK'`.

In [3]:
df_filtered = df[df['Position'] != 'GK'].copy()
print(f"Jugadores de campo después de excluir porteros: {df_filtered.shape[0]} registros")

Jugadores de campo después de excluir porteros: 15298 registros


### 4.2. Conversión de unidades: Height y Weight

El archivo original presenta:
- **Height**: cadena con formato "pies.pulgadas" (ej. "5.7" = 5 pies y 7 pulgadas). Necesitamos convertirlo a centímetros para que sea una magnitud continua y comparable.
- **Weight**: número en libras. Lo convertimos a kilogramos.

**Fórmulas:**
- 1 pie = 30.48 cm, 1 pulgada = 2.54 cm.
- 1 libra = 0.453592 kg.

In [4]:
def convert_height_to_cm(height_val):
    """Convierte formato 'pies.pulgadas' a centímetros."""
    if pd.isna(height_val):
        return np.nan
    h_str = str(height_val)
    try:
        parts = h_str.split('.')
        feet = int(parts[0])
        inches = int(parts[1]) if len(parts) > 1 else 0
        return (feet * 30.48) + (inches * 2.54)
    except:
        return np.nan

df_filtered['Height_cm'] = df_filtered['Height'].apply(convert_height_to_cm)
df_filtered['Weight_kg'] = df_filtered['Weight'] * 0.453592

print("Conversión completada. Ejemplo de valores convertidos:")
print(df_filtered[['Height', 'Height_cm', 'Weight', 'Weight_kg']].head())

Conversión completada. Ejemplo de valores convertidos:
   Height  Height_cm  Weight  Weight_kg
0    5.70     170.18     159  72.121128
1    6.20     187.96     183  83.007336
2    5.90     175.26     150  68.038800
3    5.11     180.34     154  69.853168
4    5.80     172.72     163  73.935496


### 4.3. Manejo de valores faltantes (imputación)

El dataset tiene algunos valores nulos. La estrategia definida en el flujo de trabajo es:
- Para variables con **menos del 5% de nulos**, imputamos con la **mediana por grupo posicional** (usando la columna `Position`), asumiendo que jugadores de la misma posición tienden a tener valores similares. Si tras la imputación por grupo aún quedan nulos (por ejemplo, grupos enteramente nulos), se completa con la mediana global.
- Las variables con más del 5% de nulos (si las hubiera) se imputan directamente con la mediana global.

**¿Por qué la mediana y no la media?** La mediana es robusta frente a outliers, lo cual es común en datos de jugadores (ej. algunos valores extremadamente altos en habilidades).

In [5]:
for col in selected_vars:
    if col not in df_filtered.columns:
        continue
    null_pct = df_filtered[col].isnull().mean()
    if 0 < null_pct < 0.05:
        # Imputación por mediana dentro de cada grupo de posición
        df_filtered[col] = df_filtered.groupby('Position')[col].transform(
            lambda x: x.fillna(x.median())
        )
        # Si aún quedan nulos, usar mediana global
        if df_filtered[col].isnull().sum() > 0:
            df_filtered[col] = df_filtered[col].fillna(df_filtered[col].median())

# Verificar que no queden nulos en las variables seleccionadas
X = df_filtered[selected_vars].copy()
if X.isnull().sum().any():
    print("Aún hay nulos, aplicando imputación global con mediana.")
    X = X.fillna(X.median())
else:
    print("No hay valores nulos después de la imputación por posición.")

No hay valores nulos después de la imputación por posición.


### 4.4. Estandarización (Z-score)

Los algoritmos de clustering basados en distancias (como KMeans) son sensibles a la escala de las variables. Por ello, estandarizamos todas las variables para que tengan **media 0 y desviación estándar 1**. Así, cada variable contribuye por igual a la distancia euclidiana.

Usamos `StandardScaler` de scikit-learn.

In [6]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
df_scaled = pd.DataFrame(X_scaled, columns=selected_vars, index=X.index)
print("Datos estandarizados. Forma final:", df_scaled.shape)
print("\nPrimeras 5 filas (valores estandarizados):")
df_scaled.head()

Datos estandarizados. Forma final: (15298, 35)

Primeras 5 filas (valores estandarizados):


,Age,Weak Foot,Skill Moves,Work Rate,Height_cm,Weight_kg,Crossing,Finishing,HeadingAccuracy,ShortPassing,...,Agility,Reactions,Balance,Jumping,Stamina,Strength,Aggression,Positioning,Vision,Composure
0,1.309193,1.545638,2.351377,-0.506389,-0.599361,-0.350285,2.093862,2.757318,1.110733,2.781412,...,1.986671,3.721871,2.333724,0.168754,0.403181,-0.555981,-0.816883,2.662697,2.966525,3.488564
1,1.749505,1.545638,3.963502,0.685417,0.980104,1.273427,2.093862,2.696098,2.748475,1.858798,...,1.661221,3.835917,0.275124,2.514076,1.834882,1.038164,0.224321,2.731064,2.035818,3.390203
2,0.208413,3.111854,3.963502,1.281320,-0.148085,-0.959177,1.739149,2.267552,0.421158,2.166336,...,2.393484,3.607825,1.427940,-0.439293,1.208513,-1.353054,-0.261574,2.320857,2.423613,3.291841
3,0.428569,3.111854,2.351377,1.877222,0.303191,-0.688558,2.732344,1.961449,-0.182220,2.986438,...,1.010320,3.265686,0.851532,-0.265565,2.013845,0.719335,1.126698,2.184121,2.966525,2.701672
4,0.428569,1.545638,2.351377,1.281320,-0.373723,-0.079666,1.881034,2.083890,0.334961,2.678900,...,2.312121,3.151639,2.251380,-0.873612,1.387475,0.001969,-0.400401,2.184121,2.578731,2.996756


## 5. Mapa de calor de correlaciones

Antes de aplicar PCA y clustering, es útil visualizar las correlaciones entre las variables. Un mapa de calor nos ayuda a detectar posibles multicolinealidades (variables muy correlacionadas), lo que justifica la reducción de dimensionalidad.

**Interpretación:** Colores rojos indican correlación positiva alta (ej. SprintSpeed y Acceleration), azules correlación negativa. 

In [7]:
# Calcular matriz de correlación y reformatear para Altair
corr_matrix = df_scaled.corr().reset_index().melt('index')
corr_matrix.columns = ['var1', 'var2', 'correlacion']

# Heatmap interactivo con Altair
heatmap = alt.Chart(corr_matrix).mark_rect().encode(
    x=alt.X('var1:O', title='Variable 1', sort=None),
    y=alt.Y('var2:O', title='Variable 2', sort=None),
    color=alt.Color('correlacion:Q', scale=alt.Scale(scheme='redblue', domain=[-1, 1]),
                    title='Correlación'),
    tooltip=['var1', 'var2', alt.Tooltip('correlacion:Q', format='.2f')]
).properties(
    title='Mapa de Calor de Correlaciones (Jugadores de Campo)',
    width=700,
    height=700
)

heatmap

alt.Chart(...)